# Notebook 07 — recall alignment and model diagnostics

Morning asks about **since waking up**; Midday and Endday ask about **since the last prompt**.
This notebook audits what the supplied data can establish and diagnoses constant median predictions.
It is an exploratory follow-up to Notebooks 05–06, not an independent preregistration. No new
validation prediction or score is calculated. All 32 final-test participants remain sealed.

Read the [frozen specification](../reports/recall_alignment_specification.md). Source cells display
only aggregates and clearly synthetic examples. Participant outputs remain private and ignored.
The corrected window engine, assignments, five folds and prior features are unchanged.

In [ ]:
from pathlib import Path
import json
import sys

import numpy as np
import pandas as pd
from IPython.display import Image, display

ROOT = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT / "src"))
from garmin_preprocessing.common import verify_bundle  # noqa: E402
from garmin_recall_diagnostics.recall import wake_boundary  # noqa: E402

BUNDLE = ROOT / "private/recall_diagnostics/runs/0b217d08cb1c3dce"
REPORT = ROOT / "reports/recall_alignment/824551e40b8e301d"
manifest = verify_bundle(BUNDLE)
verify_bundle(REPORT)
assert manifest["accounted_development_emas"] == 3706
assert manifest["training_fits"] == 45 and manifest["new_validation_predictions"] == 0
print({key: manifest[key] for key in ["accounted_development_emas", "boundary_rows", "candidate_rows", "training_fits", "new_validation_predictions"]})

## 1. Exact recall wording and clock availability

The linked [questionnaire evidence](../reports/recall_alignment/824551e40b8e301d/questionnaire_evidence.md) cites local blank-form
PDF pages and matching codebook fields. The forms contain outlined text and were inspected visually.
The export's stored 1–5 coding and the existing composite are unchanged. Other mood questions'
“right before starting” wording does not change these three PSS items' recall period.

Actual starts, scheduled starts and row UTC offsets are exported; wake time is a Morning answer.
There is no supplied complete prompt/delivery/submission log. A previous completed survey is not a
previous prompt. Endday does not document whole-day recall. Garmin acquisition/receipt/batching
semantics remain uncertain; no timestamps are moved to improve alignment.

In [ ]:
evidence = json.loads((REPORT / "source_evidence.json").read_text())
items = pd.DataFrame(evidence["questionnaire_items"])
with pd.option_context("display.max_colwidth", None):
    display(items[["survey_category", "export_field", "exact_wording", "rendered_form_page", "question_number"]])
display(pd.DataFrame(evidence["timing_fields"]))
print("Supplied nested member types:", evidence["structural_inventory"]["nested_member_extensions"])
print("Dedicated prompt/submission log supplied:", evidence["structural_inventory"]["dedicated_prompt_or_submission_log_supplied"])

## 2. Synthetic boundary example — no participant data

A minute-resolution start at 08:10 local has cutoff 08:09. With local-minus-UTC offset −240,
this is 12:09 UTC. A current-response wake answer of 07:00 implies a nominal *retrospective*
start at 11:00 UTC under the same-date/same-offset assumption. The later-minute sensitivity
starts at 11:01. Neither start was established as available before the current response.
Readings exactly at cutoff are excluded; partial final bins use ceil(duration/60).
Never repair a wake after cutoff by shifting it to yesterday.

In [ ]:
synthetic_actual = pd.Timestamp("2024-06-01 08:10:00")
synthetic_cutoff = pd.Timestamp("2024-06-01 12:09:00", tz="UTC").timestamp()
synthetic_start, reason = wake_boundary(synthetic_actual, -240, synthetic_cutoff, "7:00 AM")
assert reason == "candidate" and synthetic_cutoff - synthetic_start == 69 * 60
print({"synthetic_nominal_minutes": (synthetic_cutoff - synthetic_start) / 60,
       "synthetic_later_minute_minutes": (synthetic_cutoff - synthetic_start - 60) / 60,
       "known_at_prediction": False, "synthetic_20_of_69_bin_fraction": 20 / 69})
print("Occupied bins are not continuously observed physiological duration.")

## 3. Availability and matched coverage

All 3,706 responses are accounted for in each boundary scenario. Morning has 931/934 wake
answers: 836 nominal candidates, versus 826 under the later-minute sensitivity. Ninety-five
nominal wake clocks are at/after cutoff (53 after actual start, 31 equal to start, 11 within the
buffer); three are missing. No clocks are repaired. Midday/Endday remain unavailable without
prompt logs. All established wake candidates are retrospective-only; their UTC offset at waking
and same-day interpretation are assumptions.

Only 253 nominal candidates / 91 participants have any accepted HR; 543 have no recording and
40 only rejected readings. Any HR is not sufficient support for all four features or continuous
coverage. Variable-window median duration is 92 minutes. Both scenarios are compared with every
fixed lookback on the same responses, including those without HR. No new support threshold or
aligned-window model is selected.

The minute sensitivity concerns recorded granularity, not a confidence bound: recall error or rounding can exceed one minute.

In [ ]:
display(pd.read_json(REPORT / "boundary_availability.json"))
coverage = pd.read_json(REPORT / "candidate_coverage.json")
display(coverage[["scenario", "responses", "participants", "with_accepted_hr", "participants_with_accepted_hr", "no_recording", "all_readings_rejected", "duration_seconds_median", "occupied_bin_fraction_median", "maximum_gap_including_edges_seconds_median"]])
fixed = pd.read_json(REPORT / "matched_fixed_coverage.json")
display(fixed[["scenario", "lookback_minutes", "responses", "with_accepted_hr", "participants_with_accepted_hr", "occupied_bin_fraction_median"]])
display(pd.read_json(REPORT / "representation.json"))
display(Image(filename=str(REPORT / "recall_coverage.png")))

## 4. Median-regression diagnostic: training only

Use the unchanged 1,395-response / 117-participant five-minute Policy A cohort. Context, HR and
context+HR models use alpha 0, .001 and .01 in each of five original training folds: 45 fits.
Each diagnostic function receives only training rows; target reads explicitly exclude the held-out
fold. No new validation predictions or metrics. Inspecting saved prediction variation needs no
validation outcomes. Numeric scaling and category encoding are trained on each training fold only.

At quantile .5, objective = .5 × mean absolute training error + alpha × coefficient L1 norm
(intercept excluded). A constant median is always a feasible solution. Full category encoding plus
intercept has expected rank redundancy, so coefficient allocation may be nonunique. The independent
matrix/objective replay verifies numerical implementation, not physiological validity or generalization.

In [ ]:
display(pd.read_json(REPORT / "saved_model_inspection.json"))
display(pd.read_json(REPORT / "training_summary.json"))
diagnostics = pd.read_json(REPORT / "training_diagnostics.json")
assert len(diagnostics) == 45 and diagnostics.converged.all()
display(diagnostics[["heldout_fold", "model", "alpha", "training_responses", "training_participants", "nonzero_coefficients", "design_rank_with_intercept", "training_prediction_sd", "training_pinball_loss", "l1_penalty", "training_objective", "training_median_dummy_objective"]])
display(Image(filename=str(REPORT / "training_collapse.png")))

## 5. Synthetic loss example — not real outcomes

For [4,6,6,8], the median constant has MAE 1 and pinball loss .5. A hypothetical training MAE
.98 with coefficient L1 norm 2 yields objective .49 at alpha 0 and .51 at alpha .01, so the
constant remains preferable in the penalized objective. Removing the penalty permits flexibility;
it cannot manufacture a predictive relationship.

In [ ]:
synthetic_y = np.array([4., 6., 6., 8.])
synthetic_mae = np.abs(synthetic_y - np.median(synthetic_y)).mean()
assert synthetic_mae == 1.
print({"synthetic_median_mae": synthetic_mae, "synthetic_median_pinball": .5 * synthetic_mae,
       "hypothetical_unpenalized_objective": .5 * .98,
       "hypothetical_alpha_001_objective": .5 * .98 + .01 * 2})

## 6. What this resolves; next justified step

All 45 fits converge. HR-only and context-only remain constant in all folds even with no penalty.
Unpenalized context+HR varies in three folds, improving training MAE by only about .0008–.0044
composite points. This rules out penalty strength as the sole explanation for HR-only collapse;
it does not prove an absence of signal or identify a better generalizing model. No winning alpha.

Provider-only questions: complete prompt/delivery/expiry logs and intended last-prompt interpretation;
start/resume/submission definitions; wake day/offset handling; Garmin acquisition versus receipt/batching.
The questions are drafted, not sent. Choices we can test ourselves include a separately specified,
retrospective Morning-only wake-window versus fixed-window comparison on matched cohorts/folds.
The 253 with any HR are an upper bound before four-feature support checks. Keep this separate from
since-last-prompt outcomes and deployable prediction claims; do not launch an alpha search.

See the [plain-language report](../reports/recall_alignment/824551e40b8e301d/report.md),
[professor briefing](../reports/recall_alignment/824551e40b8e301d/professor_briefing.md), and
[verification](../reports/recall_alignment/verification_0b217d08cb1c3dce.md).

Reproduce calculations with `PYTHONPATH=src uv run python -m garmin_recall_diagnostics.pipeline`
after setting `GARMIN_REFERENCE_PROJECT`. The guarded entrypoint verifies/reuses immutable bundles.
Public reports can be regenerated with `garmin_recall_diagnostics.report.render(ROOT, BUNDLE)`;
independent checks use `garmin_recall_diagnostics.verify.verify(ROOT, BUNDLE, reference_path)`.
This notebook verifies the aggregate walkthrough. No participant IDs or complete participant rows
are displayed; source outputs stay empty and the executed copy belongs in ignored `outputs/`.

In [ ]:
print("Boundary accounting:", BUNDLE.relative_to(ROOT) / "boundaries.parquet")
print("Candidate coverage:", BUNDLE.relative_to(ROOT) / "candidate_coverage.parquet")
print("Training-only predictions:", BUNDLE.relative_to(ROOT) / "training_predictions.parquet")
print("Report:", REPORT.relative_to(ROOT) / "report.md")
print("Briefing:", REPORT.relative_to(ROOT) / "professor_briefing.md")